# 🚁 Notebook 07 — The Complete PID Controller

### Bringing P, I, and D together

You've built each piece separately:

- **P** (Proportional) reacts to the error **right now**.
- **I** (Integral) accumulates the error from the **past** to kill steady-state error.
- **D** (Derivative) anticipates the **future** by damping fast motion.

Put them in one controller and you get the workhorse of control engineering — used in an estimated
95%+ of industrial control loops and inside essentially every drone. This notebook assembles the
complete, reusable `PIDController` (with anti-windup and derivative filtering already built in) and
drives a **live dashboard** so you can see all three terms cooperating.

---

## 🎯 Learning objectives
- Combine P, I, D into one clean control law and a reusable controller class.
- Understand the full **controller architecture** and signal flow.
- Read a **dashboard**: altitude, velocity, thrust, and the individual P/I/D contributions.
- Compare P vs PI vs PID on the same task and see why the full PID wins.

## 🧩 Prerequisites
Notebooks 01–06 (all three terms; anti-windup; derivative filtering; the `simulate()` engine).

## ⏱️ Estimated time
About **45–60 minutes**.

## 🧠 Concepts you know so far
P, I, D individually · anti-windup · derivative filtering · steady-state error · overshoot · the engine

## 🔜 Concepts you'll learn here
The full PID law · a reusable controller class · controller architecture · a live multi-panel dashboard


### 🔁 Quick recap — the three terms in one table

| Term | Looks at | Job | Failure if overused |
|------|----------|-----|---------------------|
| **P** | error **now** | push toward target | oscillation, leftover error |
| **I** | error's **past** (sum) | erase steady-state error | windup, slow oscillation |
| **D** | error's **future** (rate) | damp / brake | amplifies noise |

The full control law simply **adds them up**:

$$ \text{thrust} = K_p\,e \;+\; K_i\!\int e\,dt \;+\; K_d\,\frac{d e}{dt} $$

Run setup and bring in the engine.


In [ ]:
# ============================================================
# SETUP CELL  -  run this first, every time
# ============================================================
import numpy as np                      # numbers and arrays
import matplotlib.pyplot as plt         # plotting / graphs
from matplotlib import animation        # for animations
from IPython.display import HTML        # to show animations in the notebook

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["animation.embed_limit"] = 80
plt.rcParams["figure.figsize"] = (8, 4)

from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete. numpy, matplotlib and ipywidgets are ready!")


### 🧰 Reusable simulator (same engine as before)

In [ ]:
from collections import deque

def simulate(controller, target=10.0, mass=1.0, g=9.8, start_height=0.0,
             dt=0.02, total_time=12.0, max_thrust=30.0, min_thrust=0.0,
             noise_std=0.0, wind=0.0, delay_steps=0, seed=0):
    """Run the 1D drone with ANY controller and return recorded signals as arrays.

    `controller` is a callable: controller(target, measured_altitude, dt) -> desired_thrust.
    It may hold internal state (integral, previous error, etc.) and may expose a dict
    `controller.last_terms = {'p':..,'i':..,'d':..}` which we record for the dashboards.

    Physical extras (all optional, default off):
      noise_std  : Gaussian sensor noise standard deviation (metres)
      wind       : constant extra force in newtons (+ up, - down)
      delay_steps: how many steps the sensor reading lags behind reality
      max/min_thrust : actuator saturation limits (newtons)
    """
    rng = np.random.default_rng(seed)
    x, v, t = start_height, 0.0, 0.0
    buf = deque([start_height] * (delay_steps + 1), maxlen=delay_steps + 1)  # sensor delay line

    keys = ("t", "x", "v", "a", "measured", "thrust", "error", "p", "i", "d")
    hist = {k: [] for k in keys}

    for _ in range(int(total_time / dt)):
        buf.append(x)                                    # newest true altitude enters the line
        delayed = buf[0]                                 # controller sees the OLD reading
        measured = delayed + (rng.normal(0, noise_std) if noise_std > 0 else 0.0)
        error = target - measured

        thrust = controller(target, measured, dt)        # <-- the controller decides
        thrust = min(max(thrust, min_thrust), max_thrust)  # actuator can't exceed limits

        net_force = thrust - mass * g + wind             # sum of forces (up +, down -)
        a = net_force / mass                             # Newton's second law

        terms = getattr(controller, "last_terms", {"p": 0.0, "i": 0.0, "d": 0.0})
        for k, val in zip(keys, (t, x, v, a, measured, thrust, error,
                                 terms.get("p", 0.0), terms.get("i", 0.0), terms.get("d", 0.0))):
            hist[k].append(val)

        v = v + a * dt                                   # Euler integrate
        x = x + v * dt
        if x < 0:                                        # ground
            x, v = 0.0, 0.0
        t = t + dt

    return {k: np.array(val) for k, val in hist.items()}


## 1. The complete, reusable `PIDController`

Here is the full controller in one clean, heavily-commented class. It bakes in everything you
learned: **anti-windup** (Notebook 05) and a **filtered derivative on the measurement**
(Notebook 06). It also records each term's contribution in `last_terms`, which our dashboard reads.

Notice there is **no gravity feed-forward** — on purpose. The **integral** term is what learns to
cancel gravity. That's why, if you set `Ki = 0`, a steady-state error reappears.


In [ ]:
class PIDController:
    """A complete PID controller with anti-windup and a filtered derivative.

    control law:  thrust = Kp*error + Ki*(integral of error) + Kd*(derivative)
    - NO gravity feed-forward on purpose, so the INTEGRAL term is what learns to
      cancel gravity. This is why removing integral leaves a steady-state error.
    - derivative is taken on the MEASUREMENT (not the error) and low-pass filtered,
      which avoids 'derivative kick' and tames sensor noise.
    """
    def __init__(self, Kp=0.0, Ki=0.0, Kd=0.0,
                 i_limit=15.0, d_filter=0.1, use_anti_windup=True):
        self.Kp, self.Ki, self.Kd = Kp, Ki, Kd
        self.i_limit = i_limit            # clamp on the integral term (anti-windup)
        self.d_filter = d_filter          # 1.0 = no smoothing, small = heavy smoothing
        self.use_anti_windup = use_anti_windup
        self.integral = 0.0
        self.prev_measured = None
        self.d_state = 0.0
        self.last_terms = {"p": 0.0, "i": 0.0, "d": 0.0}

    def __call__(self, target, measured, dt):
        error = target - measured

        # --- Proportional: react to the error RIGHT NOW ---
        p = self.Kp * error

        # --- Integral: accumulate error over time (with anti-windup clamp) ---
        self.integral += error * dt
        if self.use_anti_windup and self.Ki > 0:
            hi = self.i_limit / self.Ki
            self.integral = max(-hi, min(hi, self.integral))   # clamp integral
        i = self.Ki * self.integral

        # --- Derivative (on measurement, filtered): anticipate the future ---
        if self.prev_measured is None:
            raw = 0.0
        else:
            raw = (measured - self.prev_measured) / dt          # rate of change of altitude
        self.prev_measured = measured
        self.d_state += self.d_filter * (raw - self.d_state)    # low-pass filter
        d = -self.Kd * self.d_state                             # minus: opposes motion (damping)

        self.last_terms = {"p": p, "i": i, "d": d}
        return p + i + d


## 2. Fly it, and compare P vs PI vs PID

Same task (climb to 10 m), three controllers. Watch the story unfold:
- **P** overshoots a bit and settles short (steady-state error).
- **PI** reaches target exactly but overshoots more (no damping).
- **PID** reaches target exactly *and* glides in smoothly (damped).


In [ ]:
res_P   = simulate(PIDController(Kp=3.0, Ki=0.0, Kd=0.0), total_time=16.0)
res_PI  = simulate(PIDController(Kp=3.0, Ki=1.5, Kd=0.0), total_time=16.0)
res_PID = simulate(PIDController(Kp=3.0, Ki=1.5, Kd=3.0), total_time=16.0)

plt.figure(figsize=(9, 5))
plt.plot(res_P["t"],   res_P["x"],   color="tab:orange", lw=2, label="P    (short + overshoot)")
plt.plot(res_PI["t"],  res_PI["x"],  color="tab:red",    lw=2, label="PI   (exact, but overshoots)")
plt.plot(res_PID["t"], res_PID["x"], color="tab:blue",   lw=2, label="PID  (exact + damped)")
plt.axhline(10, color="seagreen", ls="--", label="target")
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("P vs PI vs PID on the same climb")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
print("PID = accurate (thanks to I) AND smooth (thanks to D) AND responsive (thanks to P).")


## 3. The dashboard: seeing all the signals at once

A single altitude line hides the story. A **dashboard** shows what each part is doing. Below:
**(1)** altitude vs target, **(2)** velocity, **(3)** total thrust, and **(4)** the individual
**P, I, D contributions** to that thrust. Watch how, at the end, the **I term alone holds ≈ 9.8 N**
(cancelling gravity) while P and D fade toward zero.


In [ ]:
ctrl = PIDController(Kp=3.0, Ki=1.5, Kd=3.0)
res = simulate(ctrl, total_time=16.0)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
t = res["t"]
axes[0,0].plot(t, res["x"], color="tab:blue", lw=2); axes[0,0].axhline(10, color="seagreen", ls="--")
axes[0,0].set_title("Altitude vs target"); axes[0,0].set_ylabel("m"); axes[0,0].grid(True, alpha=0.3)
axes[0,1].plot(t, res["v"], color="tab:green", lw=2); axes[0,1].axhline(0, color="gray", ls=":")
axes[0,1].set_title("Velocity"); axes[0,1].set_ylabel("m/s"); axes[0,1].grid(True, alpha=0.3)
axes[1,0].plot(t, res["thrust"], color="tab:purple", lw=2)
axes[1,0].set_title("Total thrust command"); axes[1,0].set_ylabel("N"); axes[1,0].set_xlabel("s"); axes[1,0].grid(True, alpha=0.3)
axes[1,1].plot(t, res["p"], label="P term", color="tab:orange")
axes[1,1].plot(t, res["i"], label="I term", color="tab:red")
axes[1,1].plot(t, res["d"], label="D term", color="tab:blue")
axes[1,1].axhline(9.8, color="gray", ls=":", label="m*g")
axes[1,1].set_title("P / I / D contributions"); axes[1,1].set_ylabel("N"); axes[1,1].set_xlabel("s")
axes[1,1].legend(fontsize=8); axes[1,1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("Bottom-right is the key insight: the I term rises to ~9.8 N and holds it; P and D relax to ~0.")


## 4. 🎬 Animated PID with live altitude & thrust


In [ ]:
res = simulate(PIDController(Kp=3.0, Ki=1.5, Kd=3.0), total_time=14.0)
t, x, thr = res["t"], res["x"], res["thrust"]
fids = np.linspace(0, len(t)-1, 120).astype(int)

fig, (axd, axp) = plt.subplots(1, 2, figsize=(10, 6), gridspec_kw={"width_ratios":[1, 2]})
# left: the drone
axd.set_xlim(-1, 1); axd.set_ylim(0, 13); axd.set_xticks([]); axd.set_ylabel("altitude (m)")
axd.set_title("PID drone"); axd.axhline(0, color="saddlebrown", lw=3); axd.axhline(10, color="seagreen", ls="--", lw=2)
drone, = axd.plot([], [], "o", color="tab:blue", markersize=26)
# right: live altitude trace
axp.set_xlim(0, t[-1]); axp.set_ylim(0, 13); axp.axhline(10, color="seagreen", ls="--")
axp.set_xlabel("time (s)"); axp.set_ylabel("altitude (m)"); axp.set_title("live altitude"); axp.grid(True, alpha=0.3)
trace, = axp.plot([], [], color="tab:blue", lw=2)
label = axp.text(0.5, 11.6, "", fontsize=11)

def init():
    drone.set_data([], []); trace.set_data([], []); label.set_text(""); return drone, trace, label
def update(f):
    i = fids[f]; drone.set_data([0], [x[i]]); trace.set_data(t[:i+1], x[:i+1])
    label.set_text(f"t={t[i]:.1f}s   x={x[i]:.2f} m   thrust={thr[i]:.1f} N"); return drone, trace, label
ani = animation.FuncAnimation(fig, update, frames=len(fids), init_func=init, blit=True, interval=45)
plt.close(fig); HTML(ani.to_jshtml())


## 5. 🎛️ Tune all three gains at once

This is the real deal — the interactive PID cockpit. Adjust `Kp`, `Ki`, `Kd` and watch the response
and the term-breakdown update. Some things to try:
- `Ki = 0` → steady-state error returns (settles below target).
- `Kd = 0` → overshoot and ringing return.
- Big `Kp`, no `Kd` → oscillation.
- A balanced set (e.g. Kp≈3, Ki≈1.5, Kd≈3) → fast, exact, smooth.


In [ ]:
def cockpit(Kp=3.0, Ki=1.5, Kd=3.0):
    res = simulate(PIDController(Kp, Ki, Kd), total_time=16.0)
    t = res["t"]
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    a1.plot(t, res["x"], color="tab:blue", lw=2); a1.axhline(10, color="seagreen", ls="--")
    a1.set_ylabel("altitude (m)"); a1.set_ylim(0, 15); a1.grid(True, alpha=0.3)
    sse = 10 - res["x"][int(0.9*len(res["x"])):].mean()
    a1.set_title(f"Kp={Kp}, Ki={Ki}, Kd={Kd}   |   steady-state error approx {sse:.2f} m")
    a2.plot(t, res["p"], label="P", color="tab:orange")
    a2.plot(t, res["i"], label="I", color="tab:red")
    a2.plot(t, res["d"], label="D", color="tab:blue")
    a2.set_ylabel("term (N)"); a2.set_xlabel("time (s)"); a2.legend(); a2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

interact(cockpit,
         Kp=FloatSlider(min=0.0, max=10.0, step=0.5, value=3.0),
         Ki=FloatSlider(min=0.0, max=5.0,  step=0.25, value=1.5),
         Kd=FloatSlider(min=0.0, max=10.0, step=0.5, value=3.0));


## ✅ Summary
- The complete **PID** control law adds the three terms: `thrust = Kp·e + Ki·∫e + Kd·(de/dt)`.
- A clean, reusable `PIDController` class bundles **anti-windup** and a **filtered derivative on the
  measurement** so it behaves well on real, noisy, saturating systems.
- The **dashboard** reveals the division of labour: at steady state, the **I term holds ≈ m·g**
  while P and D fade to zero.
- PID beats P and PI because it's **accurate (I)**, **smooth (D)**, and **responsive (P)** at once.


## ⚠️ Common mistakes
- **Adding I but forgetting anti-windup.** Fine in gentle sims, disastrous the moment motors saturate.
- **Adding D without filtering.** Looks great on a clean sim, screams on a noisy real sensor.
- **Assuming more of every gain is better.** Each term has a sweet spot; that's what tuning is for
  (Notebook 08).

## 🧭 Mental model
> *"PID = a spring (P) + a patient ledger (I) + a shock absorber (D), all pushing the same throttle.
> Good control is getting the three to cooperate, not compete."*

## 🌍 Real-world applications
Drone flight controllers, 3D-printer and CNC motion, car cruise/traction control, thermostats,
robot arms, and industrial process loops everywhere.


## 🧪 Exercises

**L1 — Observe.** In the dashboard (Section 3), what does the **I term** settle to, and what does
that number represent physically?
<details><summary>▸ Solution</summary>
About <b>9.8 N = m·g</b> — the exact thrust needed to hover. The integral has "learned" to cancel
gravity by itself, which is why the steady-state error is zero.
</details>

**L2 — Modify.** In the P-vs-PI-vs-PID comparison, give the PID a stronger `Kd=6`. Does overshoot
shrink? Does the rise slow down?
<details><summary>▸ Solution</summary>
Overshoot shrinks further (more damping), but the approach becomes a little slower/lazier — the
classic damping trade-off. Very large Kd would also start amplifying any noise.
</details>

**L3 — Predict.** Set `Ki=0` in the cockpit. Predict where the drone settles, then check.
<details><summary>▸ Solution</summary>
It settles **below 10 m** (steady-state error returns), because without the integral there's nothing
to supply the exact hover thrust at zero error — you're back to P/PD behaviour.
</details>


## ❓ Quick quiz
**Q1.** Which term erases steady-state error?
<details><summary>▸ Answer</summary>**I** (integral).</details>

**Q2.** Which term reduces overshoot/oscillation?
<details><summary>▸ Answer</summary>**D** (derivative).</details>

**Q3.** At a steady hover, which term is doing the heavy lifting?
<details><summary>▸ Answer</summary>**I** — it holds ≈ m·g; P and D go to zero once you're on target and still.</details>

**Q4.** Why bake anti-windup and D-filtering into the class?
<details><summary>▸ Answer</summary>So the controller behaves well on **real** systems with actuator limits and sensor noise, not just idealized sims.</details>


## 🔭 Preview of Notebook 08 — *PID Tuning*

You have a full PID controller — but how do you *choose* Kp, Ki, Kd for a new system? Notebook 08
gives you a practical, repeatable **tuning workflow** (P first, then D, then I), shows **parameter
sweeps** so you can see each gain's effect in isolation, adds **automatic performance metrics**, and
tours a classic heuristic (Ziegler–Nichols) at an intuitive level. 🚁
